In [ ]:
import polars as pl
import psycopg2
import pyarrow.parquet as pq

In [ ]:
active_genes = pl.read_parquet("./data/active_genes.parquet")
active_accs = active_genes["gene"].to_list()

# Connect to petadex
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)

# Fixed dtypes so every streamed batch below gets the exact same schema.
# Without this, a batch where a column happens to be all-null gets inferred
# as Null dtype by polars, which then conflicts with other batches when
# writing row groups to the same parquet file.
PG_TO_POLARS = {
    "text": pl.Utf8,
    "integer": pl.Int64,
    "double precision": pl.Float64,
    "numeric": pl.Float64,
    "timestamp without time zone": pl.Datetime,
}

with conn.cursor() as meta_cur:
    meta_cur.execute("""
        SELECT column_name, data_type FROM information_schema.columns
        WHERE table_name = 'sra_metadata'
        ORDER BY ordinal_position
    """)
    schema = {name: PG_TO_POLARS[dtype] for name, dtype in meta_cur.fetchall()}

In [ ]:
# Stream the table in with a server-side (named) cursor instead of fetchall(),
# so we never hold all ~8.3M rows in memory at once. Each batch gets the
# "active" column and the lat/lon filter applied, then is appended straight
# to the output parquet as its own row group.
BATCH_SIZE = 200_000
OUT_PATH = "./data/active_samples_metadata.parquet"

cur = conn.cursor(name="sra_metadata_stream")
cur.itersize = BATCH_SIZE
cur.execute("SELECT * FROM sra_metadata")

writer = None
total_rows = 0

while True:
    rows = cur.fetchmany(BATCH_SIZE)
    if not rows:
        break

    batch = pl.DataFrame(rows, schema=schema, orient="row")
    batch = batch.with_columns(
        pl.col("acc").is_in(active_accs).alias("active")
    ).filter(
        pl.col("latitude").is_not_null() & pl.col("longitude").is_not_null()
    )

    table = batch.to_arrow()
    if writer is None:
        writer = pq.ParquetWriter(OUT_PATH, table.schema)
    writer.write_table(table)

    total_rows += batch.height
    print(f"wrote {total_rows} rows so far")

if writer is not None:
    writer.close()
cur.close()

In [ ]:
conn.close()
print(f"done: {total_rows} rows written to {OUT_PATH}")